In [ ]:
!pip install vllm==0.9.2 torch==2.7.0+cu128 torchvision==0.22.0+cu128 xformers==0.0.30 transformers==4.51.3 --extra-index-url https://download.pytorch.org/whl/cu128

In [ ]:
import torch
from vllm import LLM, SamplingParams

In [ ]:
# 사용할 프롬프트 목록
prompts = [
    "대한민국의 수도는 어디인가요?",
    "LLM 서빙 최적화 기법에는 어떤 것들이 있나요?",
    "인공지능이 세상을 어떻게 바꿀까요? 한 문단으로 요약해줘.",
]

# 샘플링 파라미터 설정
# temperature: 높을수록 창의적이고 무작위적인 텍스트 생성
# top_p: 확률 분포의 누적값이 p가 될 때까지의 토큰만 고려하여 샘플링
# max_tokens: 생성할 최대 토큰 수
sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=256)

# dtype="auto"는 사용 가능한 GPU에 맞춰 자동으로 정밀도를 설정합니다. (예: float16)
llm = LLM(model="Qwen/Qwen1.5-1.8B-Chat", dtype="float16",  max_model_len=1024,   gpu_memory_utilization=0.95)
print("모델 로드가 완료되었습니다.")

In [ ]:
# 모델의 토크나이저를 가져옵니다.
tokenizer = llm.get_tokenizer()

# 각 프롬프트를 Qwen 모델의 공식 채팅 형식으로 변환합니다.
# [{"role": "user", "content": prompt}] 형식으로 대화 턴을 구성합니다.
# add_generation_prompt=True는 마지막에 모델이 답변을 생성할 차례임을 알려줍니다.
formatted_prompts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": p}],
        tokenize=False,
        add_generation_prompt=True
    ) for p in prompts
]

In [ ]:
print("Chat Template을 적용하여 추론을 시작합니다...")
outputs = llm.generate(formatted_prompts, sampling_params)
print("추론이 완료되었습니다.\n")

In [ ]:
for i, (prompt, output) in enumerate(zip(prompts, outputs)):
    generated_text = output.outputs[0].text.strip()
    print("-" * 60)
    # enumerate를 사용해 질문 번호를 붙입니다.
    print(f"✅ [질문 {i+1}]")
    print(prompt)
    print("\n✅ [답변]")
    print(generated_text)
    print("-" * 60, "\n")